# MCTNet training for Arkansas and California

Ce notebook lance automatiquement l'entrainement sur Arkansas et California,
puis sauvegarde dans Google Drive :
- un dossier de run par etat
- le meilleur checkpoint
- les metriques
- la matrice de confusion en `.npy`, `.csv` et `.png`
- un resume global `summary.json` et `summary.csv`


In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive monte.')
except ImportError:
    print('Google Colab non detecte. Adapte les chemins si besoin.')


In [ ]:
# -----------------------------
# Configuration
# -----------------------------

PROJECT_DIR = Path('/content/drive/MyDrive/New project/python')
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/mctnet_crop_mapping_2021')
PROCESSED_DIR = DRIVE_BASE_DIR / 'processed'
RUNS_DIR = PROCESSED_DIR / 'training_runs_both_states'
STATES_TO_TRAIN = ['arkansas', 'california']

EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.1
EARLY_STOPPING_PATIENCE = 15
SEED = 2021

sys.path.insert(0, str(PROJECT_DIR))
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)
print('RUNS_DIR =', RUNS_DIR)
print('STATES_TO_TRAIN =', STATES_TO_TRAIN)

for state_slug in STATES_TO_TRAIN:
    print(PROCESSED_DIR / f'{state_slug}_mctnet_dataset.npz', 'exists =', (PROCESSED_DIR / f'{state_slug}_mctnet_dataset.npz').exists())
    print(PROCESSED_DIR / f'{state_slug}_mctnet_dataset.json', 'exists =', (PROCESSED_DIR / f'{state_slug}_mctnet_dataset.json').exists())


In [ ]:
from train_mctnet_multi_state import run_multi_state_experiment
from train_mctnet import set_seed

set_seed(SEED)

args = SimpleNamespace(
    processed_dir=str(PROCESSED_DIR),
    output_dir=str(RUNS_DIR),
    states=STATES_TO_TRAIN,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    dropout=DROPOUT,
    n_stages=3,
    n_heads=5,
    kernel_size=3,
    seed=SEED,
    num_workers=0,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    cpu=False,
    no_alpe=False,
    no_mask=False,
    no_cnn=False,
    no_trans=False,
)

summary_rows = run_multi_state_experiment(
    processed_dir=PROCESSED_DIR,
    output_dir=RUNS_DIR,
    states=STATES_TO_TRAIN,
    base_args=args,
)

print(json.dumps(summary_rows, indent=2))


In [ ]:
summary_json = RUNS_DIR / 'summary.json'
summary_csv = RUNS_DIR / 'summary.csv'

print('Summary JSON:', summary_json)
print('Summary CSV:', summary_csv)
print(summary_json.read_text(encoding='utf-8'))


In [ ]:
# Les matrices de confusion seront ici:
# RUNS_DIR / 'arkansas' / 'test_confusion_matrix.png'
# RUNS_DIR / 'california' / 'test_confusion_matrix.png'
#
# Ablations possibles:
# args.no_alpe = True
# args.no_mask = True
# args.no_cnn = True
# args.no_trans = True
